# 🧪 Laboratório Prático: Mecanismo de Atenção, Temperatura e Plausibilidade
**Disciplina:** COM170 — Inteligência Artificial na Prática Acadêmica e Profissional  
**Quinzena 02 — Módulo 2:** Atenção, Temperatura e Plausibilidade (Vaswani et al., 2017; Wolfram, 2023)  
**Perfil:** Estudo Prático Guiado (Passo a Passo)

---

## 🎯 Objetivos de Aprendizagem
Neste laboratório, você investigará de forma prática e visual os pilares fundamentais da arquitetura Transformer:

1. **Mecanismo de Atenção (Self-Attention):** Como os vetores **Query ($Q$)**, **Key ($K$)** e **Value ($V$)** operam juntos para medir a relevância mútua entre tokens.
2. **Desambiguação Semântica:** Como a atenção contextualiza palavras polissêmicas (ex.: a palavra *"banco"* em *"banco da praça"* vs. *"banco digital"*).
3. **Extração de Pesos Reais de Atenção:** Como inspecionar matrizes de atenção internas de um modelo Transformer (`GPT-2`).
4. **Temperatura ($T$) e Distribuição de Probabilidades:** Como o parâmetro de temperatura altera a curva de probabilidades na escolha do próximo token (reproduzindo o experimento de Stephen Wolfram).
5. **Plausibilidade vs. Verdade:** Por que LLMs estimam sequências estatisticamente prováveis em vez de consultar fatos, e como diagnosticar alucinações.

---

## ⚙️ Pré-requisitos: Isolamento com Ambiente Virtual (`venv`)
Para manter seu Python global limpo, isole as dependências deste laboratório criando um ambiente virtual:

### 1. Criar e Ativar o Ambiente Virtual
Abra o terminal nesta pasta e execute:
```bash
# Criar a pasta do ambiente virtual (.venv)
python -m venv .venv

# Ativar o ambiente virtual:
# Windows (PowerShell):
.venv\Scripts\Activate.ps1
# Windows (Prompt de Comando CMD):
.venv\Scripts\activate.bat
# Linux / macOS:
source .venv/bin/activate
```

### 2. Instalar as Bibliotecas e Suporte ao Jupyter (`ipykernel`)
Com o ambiente ativado (você verá `(.venv)` no início da linha de comando):
```bash
pip install --upgrade pip
pip install transformers torch ipykernel
```

### 3. Selecionar o Kernel no VS Code / Jupyter
> 💡 **Dica de execução:** No canto superior direito deste notebook no VS Code, clique em **Select Kernel** (ou *Selecionar Kernel*) $\rightarrow$ **Python Environments...** $\rightarrow$ escolha o interpretador dentro de `.venv`.


## 📦 Etapa 1: Importação das Bibliotecas e Configuração do Ambiente

Importamos o PyTorch (`torch`), as funções matemáticas (`torch.nn.functional`) e as classes do Hugging Face `transformers`.

In [ ]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2Tokenizer, GPT2Model, GPT2LMHeadModel

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Versão do PyTorch: {torch.__version__}")
print(f"Dispositivo de execução: {dispositivo.upper()}")

---
## 🧠 Etapa 2: A Matemática do Mecanismo de Atenção (Query, Key, Value)

Antes do Transformer (*Vaswani et al., 2017*), os modelos liam sequencialmente token por token, sofrendo de "esquecimento de contexto" em textos longos. O mecanismo de **Self-Attention** resolve isso permitindo que cada token observe todos os demais simultaneamente.

### 📌 As Três Projeções de cada Token:
| Vetor | O que representa | Analogia Intuitiva |
| :--- | :--- | :--- |
| **Query ($Q$)** | O que este token busca no contexto. | A pergunta que o token faz aos outros. |
| **Key ($K$)** | O que este token oferece como identificação. | O cartaz/etiqueta que o token exibe. |
| **Value ($V$)** | A informação semântica que o token carrega. | O conteúdo entregue quando encontrado. |

A fórmula canônica do **Scaled Dot-Product Attention** é:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) V$$

Vamos implementar e inspecionar esse cálculo passo a passo em tensores PyTorch.

In [ ]:
# Configuração didática de tensores para 3 tokens com dimensão dk = 4
torch.manual_seed(42)
quantidade_tokens = 3
dimensao_chave = 4

# Projeções Q, K, V simuladas
matriz_query = torch.randn(quantidade_tokens, dimensao_chave)
matriz_key = torch.randn(quantidade_tokens, dimensao_chave)
matriz_value = torch.randn(quantidade_tokens, dimensao_chave)

# 1. Produto Escalar (Q * K^T): Mede a similaridade entre cada par de tokens
pontuacoes_similaridade = torch.matmul(matriz_query, matriz_key.transpose(-2, -1))

# 2. Escalonamento por sqrt(d_k): Evita gradientes muito pequenos no softmax
fator_escala = math.sqrt(dimensao_chave)
pontuacoes_escaladas = pontuacoes_similaridade / fator_escala

# 3. Softmax: Transforma as pontuações em probabilidades (pesos de atenção que somam 100% por linha)
pesos_atencao = F.softmax(pontuacoes_escaladas, dim=-1)

# 4. Multiplicação pelos Valores (V): Agregação das informações ponderadas
saida_atencao = torch.matmul(pesos_atencao, matriz_value)

print("=== MATRIZ DE PESOS DE ATENÇÃO (Probabilidades) ===")
print(pesos_atencao.round(decimals=4))
print("\nVerificação da soma de cada linha (deve ser 1.0 = 100%):")
for indice_linha, soma in enumerate(pesos_atencao.sum(dim=-1)):
    print(f"  • Linha {indice_linha + 1} (Token {indice_linha + 1}): soma = {soma.item():.4f}")

---
## 🔍 Etapa 3: Extração Real de Pesos de Atenção no GPT-2

Agora vamos extrair as matrizes de atenção reais calculadas pelas camadas internas do **GPT-2**.

Vamos passar uma frase de estudo com ambiguidade semântica:
`"The bank of the river was peaceful"`
E inspecionar como a atenção da palavra **"bank"** se distribui em relação às palavras **"river"** e **"peaceful"**.

In [ ]:
# Carregamos o modelo GPT-2 configurado para retornar todas as matrizes de atenção
tokenizador_gpt2 = GPT2Tokenizer.from_pretrained("gpt2")
modelo_gpt2_atencao = GPT2Model.from_pretrained("gpt2", output_attentions=True)

frase_teste = "The bank of the river was peaceful"
tokens_entrada = tokenizador_gpt2(frase_teste, return_tensors="pt")

with torch.no_grad():
    saida_modelo = modelo_gpt2_atencao(**tokens_entrada)

# As atenções retornadas são uma tupla com 12 camadas (layers)
# Cada camada tem formato: [batch_size, num_heads=12, seq_len, seq_len]
todas_atencoes = saida_modelo.attentions
camada_escolhida = 0   # Primeira camada (analisa relações superficiais/sintáticas)
cabeca_escolhida = 0   # Primeira cabeça de atenção

matriz_atencao_cabeca = todas_atencoes[camada_escolhida][0, cabeca_escolhida]
lista_tokens = [tokenizador_gpt2.decode([id_token]).strip() for id_token in tokens_entrada["input_ids"][0]]

print(f"Frase analisada: '{frase_teste}'")
print(f"Número de camadas (layers): {len(todas_atencoes)}")
print(f"Número de cabeças por camada: {todas_atencoes[0].shape[1]}")
print(f"Dimensão da matriz de atenção: {list(matriz_atencao_cabeca.shape)} ({len(lista_tokens)}x{len(lista_tokens)} tokens)")

print("\nPesos de atenção que o token 'bank' (índice 1) direciona aos tokens anteriores e a si mesmo:")
pesos_token_bank = matriz_atencao_cabeca[1, :2]  # No GPT-2 autorregressivo (causal), o token só olha para o passado
for indice_destino, peso in enumerate(pesos_token_bank):
    print(f"  • bank -> '{lista_tokens[indice_destino]}': {peso.item() * 100:.2f}%")

---
## 🌡️ Etapa 4: O Efeito da Temperatura na Escolha do Próximo Token

Um LLM **não pesquisa** em bancos de dados. A cada passo, ele calcula uma pontuação numérica bruta (*logits*) para cada uma das ~50.000 palavras do vocabulário e as converte em probabilidades via **Softmax com Temperatura**:

$$P(\text{token}_i) = \frac{e^{z_i / T}}{\sum_{j} e^{z_j / T}}$$

* $z_i$: *Logit* (pontuação não normalizada) do token $i$.
* $T$: Parâmetro de **Temperatura**.

### 📊 Comportamento Teórico da Temperatura:
- **$T \to 0$ (ex: $T = 0.1$):** Curva ultra concentrada no topo. Quase 100% de chance para o token nº 1 (*determinístico, conservador, ideal para código*).
- **$T \approx 0.7 - 0.8$ (média):** Equilíbrio entre plausibilidade e variedade (*recomendado por Stephen Wolfram para escrita geral*).
- **$T \ge 1.5$ (alta):** Distribuição achatada e dispersa. Palavras improváveis ganham chance relevante (*criativo, mas com alto risco de incoerência e alucinação*).

Vamos reproduzir o experimento de **Stephen Wolfram (2023)** com o prompt:
`"The best thing about AI is its ability to"`

In [ ]:
# Carregamento do modelo de linguagem causal para predição do próximo token
modelo_lm = GPT2LMHeadModel.from_pretrained("gpt2")
prompt_wolfram = "The best thing about AI is its ability to"
inputs_wolfram = tokenizador_gpt2(prompt_wolfram, return_tensors="pt")

with torch.no_grad():
    saida_lm = modelo_lm(**inputs_wolfram)

# Logits da última posição (candidatos para o próximo token imediatamente a seguir)
logits_proximo_token = saida_lm.logits[0, -1, :]

# Função para calcular o Top-K com uma determinada temperatura T
def obter_top_candidatos(logits, temperatura, k=5):
    logits_ajustados = logits / temperatura
    probabilidades = F.softmax(logits_ajustados, dim=-1)
    top_probabilidades, top_indices = torch.topk(probabilidades, k)
    
    candidatos = []
    for prob, idx in zip(top_probabilidades, top_indices):
        palavra = tokenizador_gpt2.decode([idx.item()])
        candidatos.append((palavra.strip(), prob.item() * 100))
    return candidatos

temperaturas_teste = [0.1, 0.7, 1.5]
resultados_temperatura = {t: obter_top_candidatos(logits_proximo_token, t) for t in temperaturas_teste}

print(f"Prompt: '{prompt_wolfram} [...]'\n")
print(f"{'Posição':<8} | {'T = 0.1 (Baixa)':<22} | {'T = 0.7 (Média)':<22} | {'T = 1.5 (Alta)':<22}")
print("-" * 78)

for pos in range(5):
    cand_baixa = f"'{resultados_temperatura[0.1][pos][0]}' ({resultados_temperatura[0.1][pos][1]:.1f}%)"
    cand_media = f"'{resultados_temperatura[0.7][pos][0]}' ({resultados_temperatura[0.7][pos][1]:.1f}%)"
    cand_alta  = f"'{resultados_temperatura[1.5][pos][0]}' ({resultados_temperatura[1.5][pos][1]:.1f}%)"
    print(f"#{pos+1:<7} | {cand_baixa:<22} | {cand_media:<22} | {cand_alta:<22}")

---
## 🧪 Etapa 5: Geração de Respostas Completas sob Diferentes Temperaturas

Agora vamos observar como a temperatura afeta a **geração completa de um parágrafo**.
Usaremos o modelo ultracompacto `Qwen2.5-0.5B-Instruct` gerando respostas para uma pergunta aberta sob três temperaturas distintas.

In [ ]:
nome_modelo_qwen = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_modelo_qwen)
modelo_qwen = AutoModelForCausalLM.from_pretrained(nome_modelo_qwen)

prompt_criativo = "Dê uma ideia criativa e inusitada para ensinar programação a crianças:"
inputs_qwen = tokenizador_qwen(prompt_criativo, return_tensors="pt")

for temp in [0.1, 0.7, 1.5]:
    torch.manual_seed(42)  # Fixamos a semente para comparar o impacto direto da amostragem
    with torch.no_grad():
        saida_gerada = modelo_qwen.generate(
            **inputs_qwen,
            max_new_tokens=45,
            do_sample=True,
            temperature=temp,
            pad_token_id=tokenizador_qwen.eos_token_id
        )
    
    texto_resposta = tokenizador_qwen.decode(saida_gerada[0], skip_special_tokens=True)
    print(f"==================================================")
    print(f"🌡️ TEMPERATURA = {temp}")
    print(f"==================================================")
    print(texto_resposta)
    print()


---
## 🎓 Etapa 6: Plausibilidade Não É o Mesmo que Verdade (Alucinação e Diagnóstico)

Conforme detalhado no Módulo 2 da Univesp:
> **"Atenção produz coerência, não garante verdade."**

Quando um LLM afirma algo, ele está apenas produzindo a **continuação estatisticamente mais plausível** para o texto, e não consultando uma base de fatos verificados.

### 📋 Matriz de Diagnóstico de Erros da Apostila:
| Situação | Diagnóstico Provável | O que fazer |
| :--- | :--- | :--- |
| **Modelo afirmou um fato incorreto** | Erro de plausibilidade estatística (associação frequente em treino). | Verificar em fontes primárias antes de usar. |
| **Modelo inventou citação bibliográfica** | Geração de padrão plausível sem base factual (*alucinação*). | Nunca usar citações geradas sem checar no Google Acadêmico ou Scopus. |
| **Respostas repetitivas / monótonas** | Temperatura excessivamente baixa ($T \le 0.1$). | Aumentar ligeiramente a temperatura ou reformular o prompt. |
| **Contradição no final de um texto longo** | Perda de força da atenção por limite da janela de contexto. | Dividir a tarefa em partes menores e modulares. |

---

## 📝 Roteiro de Autoavaliação e Fixação
1. **Mecanismo de Atenção:** Por que os vetores Query e Key precisam ter a mesma dimensão ($d_k$), enquanto o vetor Value pode ter dimensão diferente?
2. **Temperatura:** O que acontece numericamente com a fração $\frac{z_i}{T}$ quando a temperatura $T$ tende a zero? Por que isso torna a saída determinística?
3. **Uso Acadêmico:** Por que citar um modelo de linguagem como fonte de um fato histórico ou científico é considerado um erro metodológico grave?